# 21 — Map a query arm onto the scPoli atlas (papermill-parameterized)

In [ ]:
from pathlib import Path
import sys
import time
import json

sys.path.insert(0, str(Path.cwd().parent))
from src.paths import DATA_OUT, MODEL_OUT
import anndata as ad
from scarches.models.scpoli import scPoli

In [ ]:
# --- PARAMETERS (overridden by papermill) ---
QUERY_SOURCE = "source_8"
T_ARM = "tplus"
REFERENCE_ATLAS = "excl_source_8"   # "excl_{source}" for Exp1; "scenario_5_full" for Exp2
REFERENCE_SCENARIO = "scenario_5"   # used to look up shared compounds for labeled_indices
N_EPOCHS = 50
PRETRAIN_EPOCHS = 0
# --- END PARAMETERS ---

## Load atlas + query

In [ ]:
atlas_dir = MODEL_OUT / f"scpoli_atlas_{REFERENCE_ATLAS}"
query = ad.read_h5ad(DATA_OUT / "query_arms" / f"{QUERY_SOURCE}_{T_ARM}.h5ad")
print(f"atlas: {atlas_dir}, query: {query.shape}")

## scArches load_query_data + fine-tune

In [ ]:
import pandas as pd
import numpy as np
from src.paths import scenario_input_parquet
ref_compounds = set(
    pd.read_parquet(scenario_input_parquet(REFERENCE_SCENARIO), columns=["Metadata_JCP2022"])
    ["Metadata_JCP2022"].unique()
)
labeled_mask = query.obs["Metadata_JCP2022"].isin(ref_compounds)
labeled_indices = list(np.where(labeled_mask)[0])
print(f"labeled_indices: {len(labeled_indices)}/{len(query)} wells share compounds with reference")

model_query = scPoli.load_query_data(
    adata=query,
    reference_model=str(atlas_dir),
    labeled_indices=labeled_indices,
)

t0 = time.perf_counter()
model_query.train(
    n_epochs=N_EPOCHS,
    pretraining_epochs=PRETRAIN_EPOCHS,
    eta=10,
)
elapsed = time.perf_counter() - t0
print(f"mapping took {elapsed:.1f}s")

In [ ]:
model_query.model.eval()
query.obsm["X_scpoli_mapped"] = model_query.get_latent(query, mean=True)

## Persist

In [ ]:
out_dir = DATA_OUT / "mapped"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{QUERY_SOURCE}_{T_ARM}_scpoli.h5ad"
query.write_h5ad(out_path)
(out_dir / f"{QUERY_SOURCE}_{T_ARM}_scpoli_timings.json").write_text(
    json.dumps({"mapping_seconds": elapsed, "n_epochs": N_EPOCHS})
)
print(f"wrote {out_path}")

# Save fine-tuned query model so mapping can be reproduced without retraining
model_save_path = out_dir / f"{QUERY_SOURCE}_{T_ARM}_scpoli_model"
model_query.save(str(model_save_path), overwrite=True)
print(f"saved query model to {model_save_path}")